
<a id='rag'></a>

# Retrieval-Augmented Generation (RAG)

## RAG på norsk: Gjenfinningsforsterket tekstgenerering


## Språkmodellen

Vi kommer til å bruke modeller fra [Ollama](https://ollama.com/), en kjent plattform for modeller som kan brukes både på lokal maskin og i skyløsninger. I denne oppgaven vil vi bruke LLM [gemma3:1b](https://ollama.com/library/gemma3), som er en familie av modeller fra Google DeepMind. Dette er en liten modell med bare 1 milliard parametere. Det bør være mulig å bruke den på de fleste bærbare maskiner.

In [1]:
pwd

'/Users/ragnhildsundsbak/rtd-tutorial/docs/source/notebooks'

In [2]:
import os
print(os.getcwd())
print(output_folder)

/Users/ragnhildsundsbak/rtd-tutorial/docs/source/notebooks


NameError: name 'output_folder' is not defined

In [3]:
import os
# os.environ['HF_HOME'] = '/Users/ragnhildsundsbak/from-gutenberg-to-rstudio'
# vi må ha os på grunn av tokenet fra HF

import torch
device = 0 if torch.cuda.is_available() else -1

from langchain_community.llms import Ollama

llm = Ollama(
    model="mistral:latest"
)

query = 'What are the major contributions of the Trivandrum Observatory?'
output = llm.invoke(query)
print(output)

/var/folders/t3/8s0ymlkd615fnztk4s7kl2qh0000gn/T/ipykernel_1804/1203864120.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama
/var/folders/t3/8s0ymlkd615fnztk4s7kl2qh0000gn/T/ipykernel_1804/1203864120.py:10: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


 The Trivandrum Observatory, also known as the Thiruvananthapuram Observatory, located in India, has made significant contributions to astronomy and space science over the years. Here are some key achievements:

1. Discovery of comets: The observatory has been instrumental in discovering several comets, such as Comet C/1967 T1 (Kohoutek) in 1967, Comet 252P/Linear-Kossof in 1983, and Comet P/1996 R2 (LINEAR) in 1996.

2. Asteroid discoveries: Astronomers at the Trivandrum Observatory have discovered several near-Earth asteroids, including 4079 Thiruvananthapuram in 1988 and 5783 Vavilov in 1991.

3. Variable star research: The observatory has contributed to the study of variable stars, which are stars that change their brightness over time. They have cataloged thousands of variable stars and discovered several new ones, including RV Tauri variables.

4. Astrometric measurements: The observatory's accurate astrometric measurements have been crucial in determining the positions and movem

In [6]:
from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(
    model="granite-embedding:latest",
)

document_folder = '/Users/ragnhildsundsbak/rtd-litteratur'

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Print length of documents before embedding
documents = []
for filename in os.listdir(document_folder):
    if filename.endswith(".pdf"):
        path = os.path.join(document_folder, filename)
        loader = PyPDFLoader(path)
        documents.extend(loader.load())

print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:500])

print(f'Number of documents:', len(documents))
print('Maximum document length: ', max([len(doc.page_content) for doc in documents]))

74
{'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5.5 (7.5.3)', 'creationdate': '2023-07-11T19:51:34+05:30', 'moddate': '2023-07-11T19:51:35+05:30', 'trapped': '/False', 'source': '/Users/ragnhildsundsbak/rtd-litteratur/hess_2023.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}
Simulations and Active Learning in the Asian Studies 
Classroom: A Look at Model Diplomacy
Steve Hess, Transylvania University, US, shess@transy.edu
This paper reviews the literature on simulations-based teaching in the discipline of international 
relations and associated social science fields, tracing the development of frequently used simulations 
platforms over the last half-century. It then examines the application of the Council on Foreign 
Relations’ Model Diplomacy  program in three cour
Number of documents: 74
Maximum document length:  7166


In [8]:
# Print one of the documents
print(documents[0])

page_content='Simulations and Active Learning in the Asian Studies 
Classroom: A Look at Model Diplomacy
Steve Hess, Transylvania University, US, shess@transy.edu
This paper reviews the literature on simulations-based teaching in the discipline of international 
relations and associated social science fields, tracing the development of frequently used simulations 
platforms over the last half-century. It then examines the application of the Council on Foreign 
Relations’ Model Diplomacy  program in three courses, International Crisis Simulations, Political 
Development, and Politics of Asia, at a small liberal arts college in the South from 2018 to 2019 and 
considers the effectiveness of simulations-based teaching in achieving desired learning outcomes, 
such as critical and analytical thinking, oral and written communication, and collaboration. Finally, 
the paper provides practical steps and suggestions for the integration of Model Diplomacy and other 
simulations into an array of A

In [9]:
# Chunking and embedding Printing new length, this time of chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700, #  Could be more, for larger models like mistralai/Ministral-8B-Instruct-2410
    chunk_overlap  = 200,
)
documents = text_splitter.split_documents(documents)

print(f'Antall dokuemnt-chunks etter splitting:', len(documents))
print('Maximum document length: ', max([len(doc.page_content) for doc in documents]))

Antall dokuemnt-chunks etter splitting: 489
Maximum document length:  700


In [10]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, ollama_embeddings)

In [11]:
# første gang man lager embedding:
#import os
#import torch

#from langchain_community.document_loaders import PyPDFLoader
#from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_ollama import OllamaEmbeddings
#from langchain_community.vectorstores import FAISS

# Embeddings fra Ollama
ollama_embeddings = OllamaEmbeddings(
    model="granite-embedding:latest",
)

# 1. Last alle PDF-er fra mappen
document_folder = "/Users/ragnhildsundsbak/rtd-litteratur"

# 3. Bygg FAISS-vektorstore fra dokumenter (embedder + indekserer)
print("Genererer embeddings og oppretter FAISS-indeks...")
vectorstore = FAISS.from_documents(documents, ollama_embeddings)

# 4. Lagre FAISS-indeksen lokalt
output_folder = "data/embeddings"  # velg selv hvor i prosjektet
os.makedirs(output_folder, exist_ok=True)

print(f"Lagrer FAISS-indeksen til mappen: {output_folder}/")
vectorstore.save_local(output_folder)
print("Lagring fullført!")


Genererer embeddings og oppretter FAISS-indeks...
Lagrer FAISS-indeksen til mappen: data/embeddings/
Lagring fullført!


In [12]:
# Senere, legger til nye PDFer

#import os

#from langchain_community.document_loaders import PyPDFLoader
#from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_ollama import OllamaEmbeddings
#from langchain_community.vectorstores import FAISS

# Samme embedding-modell som før
ollama_embeddings = OllamaEmbeddings(
    model="granite-embedding:latest",
)

# 1. Last eksisterende FAISS-indeks fra disk
output_folder = "data/embeddings"

print(f"Laster eksisterende FAISS-indeks fra: {output_folder}/")
vectorstore = FAISS.load_local(
    output_folder,
    ollama_embeddings,
    allow_dangerous_deserialization=True,  # ofte nødvendig i nyere langchain-versjoner
)
print("Indeks lastet!")

# 2. Finn nye PDF-er (her må du selv definere hva som er "nytt":
#    - annen mappe,
#    - eller holde en liste/logg over allerede behandlede filer).
document_folder = "/Users/ragnhildsundsbak/rtd-tutorial/data/pdf/new_pdfs"

new_documents = []
for filename in os.listdir(document_folder):
    if filename.endswith(".pdf"):
        path = os.path.join(document_folder, filename)
        # Her kunne du filtrert bort filer du vet er med fra før
        loader = PyPDFLoader(path)
        new_documents.extend(loader.load())

print(f"Nye sider (før splitting): {len(new_documents)}")

# 3. Splitt de nye dokumentene i chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
)
new_documents = text_splitter.split_documents(new_documents)

print(f"Nye dokument-chunks (etter splitting): {len(new_documents)}")

# 4. Legg til de nye chunksene i eksisterende indeks (append)
print("Legger til nye dokumenter i eksisterende FAISS-indeks...")
vectorstore.add_documents(new_documents)
print("Ferdig med å legge til.")

# 5. Lagre indeksen på nytt (samme mappe/filnavn)
print(f"Lagrer oppdatert FAISS-indeks til: {output_folder}/")
vectorstore.save_local(output_folder)
print("Oppdatert indeks lagret.")


Laster eksisterende FAISS-indeks fra: data/embeddings/
Indeks lastet!
Nye sider (før splitting): 93
Nye dokument-chunks (etter splitting): 609
Legger til nye dokumenter i eksisterende FAISS-indeks...
Ferdig med å legge til.
Lagrer oppdatert FAISS-indeks til: data/embeddings/
Oppdatert indeks lagret.


In [14]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [15]:
from langchain_classic.prompts import PromptTemplate

prompt_template = '''You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
Context: {context}

Question: {input}

Answer:
'''

prompt = PromptTemplate(template=prompt_template,
                        input_variables=['context', 'input'])

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

combine_documents_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_documents_chain)

result = rag_chain.invoke({'input': query})

print(result['answer'])

 The Trivandrum Observatory has made significant contributions in the field of astronomy, notably through its independent observations. In 1843 and 1844, it calculated a comet, marking one of its earliest achievements. A century later, in 1941, another comet was observed independently by the observatory.

In recent times, after a major overhaul of its facilities starting in 2017, the institution achieved first light in 2022. In 2023, it successfully tracked the comet C/2023(ZTF), showcasing its present capabilities. More recently, the comet C/2023 A3 (Tsuchinshan-ATLAS) was also tracked by the observatory.

Despite facing challenges such as light pollution, incessant weather conditions, and technology, the Trivandrum Observatory continues to strive as a beacon of knowledge, reflecting its founder's vision. However, it's important to note that this information is specific to the contributions made by the Trivandrum Observatory, and comparisons with other observatories like the Madras Ob

Får du feilmeldinger? Lant ned [Sublime text](https://www.sublimetext.com/download) slik at du lettere får oversikt over koden din. Output filen gir et linjenummer for der feilen ligger. Da kan du lese av rette linjenummeret i Sublime editoren.